# Clustering with K-means
In this demonstration, we will test some the k-means algorithm with network traffic. Specifically, we will use such a clustering technique to group network flows into clusters in an unsupervised fashion.

We will use a dataset of benign and various DDoS attacks from the CIC-DDoS2019 dataset (https://www.unb.ca/cic/datasets/ddos-2019.html).
The network traffic has been previously pre-processed in a way that packets are grouped in bi-directional traffic flows using the 5-tuple (source IP, destination IP, source Port, destination Port, protocol). Each flow is represented with 21 packet-header features computed from max 1000 packets:

| Feature nr.         | Feature Name |
|---------------------|---------------------|
| 00 | timestamp (mean IAT) | 
| 01 | packet_length (mean)| 
| 02 | IP_ttl (mean) |
| 03 | IP_flags_df (sum) |
| 04 | IP_flags_mf (sum) |
| 05 | IP_flags_rb (sum) | 
| 06 | IP_frag_off (sum) |
| 07 | TCP_length (mean) |
| 08 | TCP_flags_ack (sum) |
| 09 | TCP_flags_cwr (sum) |
| 10 | TCP_flags_ece (sum) |
| 11 | TCP_flags_fin (sum) |
| 12 | TCP_flags_push (sum) |
| 13 | TCP_flags_reset (sum) |
| 14 | TCP_flags_syn (sum) |
| 15 | TCP_flags_urg (sum) |
| 16 | TCP_window_size (mean) |
| 17 | UDP_length (mean) |
| 18 | ICMP_type (mean) |
| 19 | ICMP_code (mean) |
| 20 | Packets (counter)|

In [ ]:
# Author: Roberto Doriguzzi-Corin
# Project: Course on Network Intrusion and Anomaly Detection with Machine Learning
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#   http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import os
import numpy as np
import matplotlib.pyplot as plt
from util_functions import *
from sklearn.cluster import KMeans

DATASET_FOLDER1 = "00-WebDDoS"
DATASET_FOLDER2 = "04-UDPLag"

X_train1, y_train1 = load_dataset(DATASET_FOLDER1 + "/*" + '-train.hdf5')
X_train2, y_train2 = load_dataset(DATASET_FOLDER2 + "/*" + '-train.hdf5')




SEED=1
feature_names = get_feature_names(flatten=True)
target_names = [DATASET_FOLDER1, DATASET_FOLDER2]



# Dataset re-labelling
The samples of the two attacks are re-labelled to allow for a clustering of the flows into two clusters. 

In [ ]:
# Extract the attack samples (y_train1 == 1, y_train2 == 1), relabel them to 1,2 respectively, and combine them into a single dataset
# First we need to relabel the attack samples in each dataset
y_train1_relabel = np.where(y_train1 == 1, 1, 0)  # WebDDoS attacks are labeled as 1
y_train2_relabel = np.where(y_train2 == 1, 2, 0)  # DNS attacks are labeled as 2

# Now we can combine the datasets
X_combined = np.concatenate((X_train1[y_train1_relabel == 1],
                             X_train2[y_train2_relabel == 2]), axis=0)
y_combined = np.concatenate((y_train1_relabel[y_train1_relabel == 1],
                             y_train2_relabel[y_train2_relabel == 2]), axis=0)


# Dataset analysis
We first plot the distribution of the features to understand why the effectiveness of clustering strongly depends on the choice of the input features. Recall that each sample can be represented as a point in an ```n```-dimensional feature space, where ```n``` is the number of features. Clustering algorithms partition the samples according to their distances or similarities in this feature space. Consequently, if the selected features do not capture the underlying structure of the data, samples belonging to different classes may overlap, making it difficult or even impossible for the clustering algorithm to identify meaningful groups. Conversely, choosing informative and discriminative features results in well-separated clusters, enabling the algorithm to distinguish between different classes more effectively.

In [ ]:
# Loop across the features and plot the distribution of each feature for benign and ddos traffic
for i in range(X_combined.shape[1]):
    feature_name = feature_names[i]
    webddos_values = X_combined[y_combined == 1, i]
    dns_values = X_combined[y_combined == 2, i]

    plt.figure(figsize=(10, 5))
    plt.hist(webddos_values, bins=50, alpha=0.5, label=target_names[0], color='blue', density=True)
    plt.hist(dns_values, bins=50, alpha=0.5, label=target_names[1], color='orange', density=True)
    plt.title(f'Distribution of Feature: {feature_name}')
    plt.xlabel('Feature Value')
    plt.ylabel('Density')
    plt.legend()
    plt.grid()
    plt.show()

## Plot the points in a 2D feature space
For visualizazion purposes, we select only two features to plot the distribution of benign and ddos traffic in a 2D space.

In [ ]:
# Select only the sencond and the third features for visualization purposes
feature_1 = 'IP_ttl'
feature_2 = 'packet_length'

X_train_2D = X_combined[:, [feature_names.index(feature_1), feature_names.index(feature_2)]]

# Visualize the three classes in a 2D scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(X_train_2D[y_combined == 1, 0], X_train_2D[y_combined == 1, 1], label=target_names[0], marker='o', color='blue', edgecolor='k', s=100)
plt.scatter(X_train_2D[y_combined == 2, 0], X_train_2D[y_combined == 2, 1], label=target_names[1], marker='s', color='yellow', edgecolor='k', s=100)
plt.xlabel(feature_1)
plt.ylabel(feature_2)
plt.legend()        

## Clustering with k-Means

In [ ]:
k = len(target_names)
kmeans = KMeans(n_clusters=k, random_state=SEED) # The SEED is used for reproducibility, ensuring that the same initial centroids are chosen each time the algorithm is run.
y_pred_kmeans = kmeans.fit_predict(X_train_2D)

In [ ]:
# Visualize the clusters in a 2D scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(X_train_2D[:, 0], X_train_2D[:, 1], c=y_pred_kmeans, cmap='viridis', marker='o', edgecolor='k', s=100)
plt.title('KMeans Clustering')
plt.xlabel(feature_1)
plt.ylabel(feature_2)       
plt.colorbar(label='Cluster Label')
plt.show()

## K-means accuracy
k-means assign a cluster label to each data point, but these labels do not necessarily correspond to the true class labels. To evaluate the performance of K-Means, we can use metrics such as Adjusted Rand Index (ARI), which compare the predicted cluster labels with the true class labels without requiring a one-to-one mapping between them.

In [ ]:
from sklearn.metrics import adjusted_rand_score 

ari = adjusted_rand_score(y_combined, y_pred_kmeans)

print(f"Adjusted Rand Index (ARI): {ari}")

## Evaluation metrics: WCSS
Now, we evaluate the quality of the clusters with the Within-Cluster Sum of Squares (WCSS) metric. WCSS is a measure of how tightly the clusters are packed. It is calculated as the sum of squared distances between each point and the centroid of its assigned cluster. A lower WCSS indicates that the clusters are more compact and well-defined.

In [ ]:
wcss = kmeans.inertia_
print(f"Within-Cluster Sum of Squares (WCSS): {wcss}")

## Optimal number of clusters with the "elbow" method

In [ ]:
# Optimal number of clusters using the Elbow Method
wcss_values = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, random_state=SEED,n_init=5)
    kmeans.fit(X_train_2D)
    wcss_values.append(kmeans.inertia_) 

# Plot the WCSS values to visualize the Elbow Method
plt.figure(figsize=(8, 6))
plt.plot(range(1, 11), wcss_values, marker='o')
plt.title('Elbow Method for Optimal Number of Clusters')
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS (Within-Cluster Sum of Squares)')
plt.xticks(range(1, 11))
plt.grid()
plt.show()
